# Load a Model from Hugging Face

This tutorial shows how to download and load pretrained models from the Hugging Face Hub into Julia using `Transformers.jl` and `HuggingFaceHub.jl`.

By the end you will know how to:
- Install the required packages
- Authenticate for gated models
- Load a tokenizer and a model
- Inspect the model configuration
- Save and reload from a local directory

## 1. Install Packages

If you have not installed these packages yet, run the cell below once. It is safe to re-run.

In [ ]:
Pollis.packages("Transformers", "HuggingFaceHub", "JSON3")

In [ ]:
using Pkg
Pkg.add(["Transformers", "HuggingFaceHub"])

## 2. Load a Public Model

We start with `bert-base-uncased`, a widely-used encoder model that requires no authentication.

The first call downloads the weights to `~/.cache/huggingface/hub/` and subsequent calls load from cache.

In [ ]:
using Transformers, Transformers.HuggingFace

# Load tokenizer
tkr = HuggingFace.load_tokenizer("bert-base-uncased")

# Load model — pass the architecture type as the first argument
model = HuggingFace.load_model(HuggingFace.BertModel, "bert-base-uncased")

println("Tokenizer vocab size: ", length(tkr.vocab))
println("Model hidden size:    ", model.config.hidden_size)

## 3. Choose the Right Architecture Type

The architecture type constant tells Transformers.jl which struct to deserialise the weights into. Common ones:

| Model family | Type constant |
|---|---|
| BERT / RoBERTa / DistilBERT | `HuggingFace.BertModel` |
| BERT sequence classification | `HuggingFace.BertForSequenceClassification` |
| BERT token classification | `HuggingFace.BertForTokenClassification` |
| GPT-2 | `HuggingFace.GPT2Model` |
| Llama 2 / 3 / Gemma | `HuggingFace.LlamaModel` |
| Mistral / Zephyr | `HuggingFace.GPTNeoXModel` |
| T5 / Flan-T5 | `HuggingFace.T5Model` |
| Whisper | `HuggingFace.WhisperForConditionalGeneration` |
| ViT | `HuggingFace.ViTModel` |
| CLIP | `HuggingFace.CLIPModel` |

If you are unsure, check the `architectures` field in the model's `config.json` on the Hub.

## 4. Load a Specific Revision

You can pin to a branch, tag, or commit SHA to ensure reproducibility.

In [ ]:
# Pin to a specific commit SHA for reproducibility
tkr_pinned = HuggingFace.load_tokenizer("bert-base-uncased"; revision="a265f773a47193eed794233aa2a0f0bb6d3aba85")
model_pinned = HuggingFace.load_model(HuggingFace.BertModel, "bert-base-uncased"; revision="a265f773a47193eed794233aa2a0f0bb6d3aba85")

println("Loaded pinned revision")

## 5. Authenticate for Gated Models

Some models (Llama 3, Gemma, Mistral) require you to accept a licence on the Hub and then authenticate with a token.

1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a **Read** token.
2. Accept the model licence on its Hub page.
3. Set the token as an environment variable before loading.

In [ ]:
# Set your token — do not hard-code it in notebooks you share
ENV["HF_TOKEN"] = "hf_..."   # replace with your token

tkr_llama = HuggingFace.load_tokenizer("meta-llama/Llama-3.2-3B-Instruct")
model_llama = HuggingFace.load_model(HuggingFace.LlamaModel, "meta-llama/Llama-3.2-3B-Instruct")

println("Llama hidden size: ", model_llama.config.hidden_size)

## 6. Load a Vision Processor

Vision and audio models use a *processor* instead of a tokenizer. The processor handles image resizing, normalisation, and conversion to the tensor format the model expects.

In [ ]:
# Vision Transformer — image classification
proc_vit = HuggingFace.load_processor("google/vit-base-patch16-224")
model_vit = HuggingFace.load_model(HuggingFace.ViTForImageClassification, "google/vit-base-patch16-224")

println("ViT num labels: ", length(model_vit.config.id2label))

## 7. Inspect the Model Configuration

Every loaded model exposes a `config` field that mirrors the Hub's `config.json`. It is the authoritative source for architecture hyperparameters.

In [ ]:
cfg = model.config

println("hidden_size:         ", cfg.hidden_size)
println("num_hidden_layers:   ", cfg.num_hidden_layers)
println("num_attention_heads: ", cfg.num_attention_heads)
println("intermediate_size:   ", cfg.intermediate_size)
println("vocab_size:          ", cfg.vocab_size)

## 8. Download Files Without Loading a Model

Sometimes you only need the raw files — for example to inspect `config.json` or download GGUF weights for Ollama.

In [ ]:
using HuggingFaceHub

# Download a single file; returns its local cache path
config_path = HuggingFaceHub.hf_hub_download("bert-base-uncased", "config.json")
println("config.json cached at: ", config_path)

# Read it directly
using JSON3
config_json = JSON3.read(read(config_path, String))
println("model_type: ", config_json.model_type)

## 9. Save and Reload from a Local Directory

Once you have a trained or fine-tuned model, save it locally so it can be reloaded without re-downloading from the Hub.

In [ ]:
using Transformers.HuggingFace

# Save tokenizer and model to a local folder
HuggingFace.save_tokenizer(tkr, "saved_model/")
HuggingFace.save_model(model, "saved_model/")

# Reload — identical API to loading from the Hub
tkr2   = HuggingFace.load_tokenizer("saved_model/")
model2 = HuggingFace.load_model(HuggingFace.BertModel, "saved_model/")

println("Reloaded vocab size: ", length(tkr2.vocab))

## 10. Work Offline

Once files are cached, you can work without a network connection by setting `HF_HUB_OFFLINE`.

In [ ]:
# Prevent any network calls — load from cache only
ENV["HF_HUB_OFFLINE"] = "1"

tkr_offline   = HuggingFace.load_tokenizer("bert-base-uncased")
model_offline = HuggingFace.load_model(HuggingFace.BertModel, "bert-base-uncased")

println("Loaded from cache successfully")

# Restore normal mode
delete!(ENV, "HF_HUB_OFFLINE")

## Summary

| Task | Function |
|---|---|
| Load tokenizer | `HuggingFace.load_tokenizer(model_id)` |
| Load model | `HuggingFace.load_model(ArchType, model_id)` |
| Load processor (vision/audio) | `HuggingFace.load_processor(model_id)` |
| Pin revision | add `; revision="sha_or_tag"` |
| Authenticate | set `ENV["HF_TOKEN"]` |
| Download file only | `HuggingFaceHub.hf_hub_download(id, filename)` |
| Save locally | `HuggingFace.save_tokenizer` / `save_model` |
| Work offline | set `ENV["HF_HUB_OFFLINE"] = "1"` |

Once the model is loaded, continue with **Tutorial 02 — Run Inference** to put it to work.